# AI Career Path Recommender for Tier-2/3 City Job Seekers
**IBM SkillsBuild | AI & Machine Learning Internship Project | UN SDG 8: Decent Work and Economic Growth**

### Problem statement
Many skilled individuals in Tier-2/3 cities lack guidance on which job roles match their skills. This project builds a simple AI tool that takes a user's skills and interests and suggests suitable career paths along with the skills they still need to learn.

### Approach
1. Build a small dataset of job roles and the skills each role needs.
2. Convert skills into numeric vectors using **TF-IDF** (rare, specialised skills get more weight than common ones).
3. Compare the user's skill vector with every role using **cosine similarity**.
4. Combine similarity with skill coverage (and an optional interest-area bonus) into a match percentage.
5. Show the top 3 roles and the **skills to learn next** for each.

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Job roles dataset
A small hand-built dataset of 20 entry-level roles. Each role has an interest area and a comma-separated list of required skills. It is saved as `job_roles.csv`, so you can add or edit roles easily.

In [ ]:
roles = [
    ("Data Analyst", "Data Analytics", "excel, sql, python, data visualization, statistics, power bi"),
    ("Data Scientist", "Data Science & AI", "python, machine learning, statistics, pandas, sql, data visualization"),
    ("Machine Learning Engineer", "Data Science & AI", "python, machine learning, deep learning, tensorflow, git, docker"),
    ("Computer Vision Engineer", "Data Science & AI", "python, opencv, deep learning, machine learning, numpy"),
    ("Python Backend Developer", "Software Development", "python, fastapi, sql, rest api, git, docker"),
    ("Java Developer", "Software Development", "java, sql, spring boot, rest api, git, oop"),
    ("Mobile App Developer", "Software Development", "kotlin, flutter, rest api, git, ui design"),
    ("Frontend Developer", "Web Development", "html, css, javascript, react, git, responsive design"),
    ("Full Stack Developer", "Web Development", "html, css, javascript, react, node.js, sql, git"),
    ("Software Tester (QA)", "Quality Assurance", "manual testing, selenium, sql, test cases, java"),
    ("DevOps Engineer", "Cloud & DevOps", "linux, docker, git, ci/cd, cloud computing, shell scripting"),
    ("Cloud Engineer", "Cloud & DevOps", "cloud computing, linux, networking, docker, python"),
    ("Cybersecurity Analyst", "Security", "networking, linux, cybersecurity, python, risk assessment"),
    ("Database Administrator", "Data & Databases", "sql, database design, backup and recovery, linux, performance tuning"),
    ("Data Engineer", "Data & Databases", "python, sql, etl, apache spark, cloud computing, data warehousing"),
    ("Business Analyst", "Business", "excel, sql, communication, documentation, power bi, problem solving"),
    ("UI/UX Designer", "Design", "figma, ui design, user research, wireframing, prototyping"),
    ("Digital Marketing Executive", "Marketing", "seo, social media, content writing, google analytics, communication"),
    ("Content Writer", "Marketing", "content writing, seo, communication, research, editing"),
    ("Technical Support Engineer", "IT Support", "troubleshooting, networking, communication, windows, linux"),
]

jobs = pd.DataFrame(roles, columns=["role", "interest_area", "required_skills"])
jobs.to_csv("job_roles.csv", index=False)

print("Roles:", len(jobs))
print("Interest areas:", sorted(jobs["interest_area"].unique()))
display(jobs.head())

## 3. Convert skills to vectors (TF-IDF)
Skills are separated by commas, so multi-word skills like *machine learning* stay as one unit instead of being split into two words.

In [ ]:
def skill_tokenizer(text):
    return [s.strip().lower() for s in text.split(",") if s.strip()]

vectorizer = TfidfVectorizer(tokenizer=skill_tokenizer, token_pattern=None, lowercase=False)
role_matrix = vectorizer.fit_transform(jobs["required_skills"])
known_skills = set(vectorizer.get_feature_names_out())

print("Unique skills known to the model:", len(known_skills))
print("Role matrix shape (roles x skills):", role_matrix.shape)

## 4. The recommender
**Score for each role** = 60% cosine similarity + 40% skill coverage (share of the role's skills the user already has), plus a small +5% bonus if the role is in the user's chosen interest area. The result is shown as a match percentage.

Common shorthand such as `ml`, `js` or `powerbi` is mapped to the full skill name first.

In [ ]:
ALIASES = {
    "ml": "machine learning", "dl": "deep learning", "js": "javascript",
    "powerbi": "power bi", "power-bi": "power bi", "nodejs": "node.js", "node": "node.js",
    "cv": "opencv", "tf": "tensorflow", "rest": "rest api", "restapi": "rest api",
    "cloud": "cloud computing", "aws": "cloud computing", "azure": "cloud computing",
    "dataviz": "data visualization", "visualization": "data visualization",
    "spring": "spring boot", "ui": "ui design", "testing": "manual testing",
    "ci": "ci/cd", "cicd": "ci/cd", "spark": "apache spark",
}

def clean_skills(raw):
    skills = []
    for s in raw.split(","):
        s = s.strip().lower()
        if s:
            skills.append(ALIASES.get(s, s))
    return list(dict.fromkeys(skills))   # remove duplicates, keep order


def recommend(user_skills, interest=None, top_n=3):
    skills = clean_skills(user_skills)
    unknown = [s for s in skills if s not in known_skills]
    if unknown:
        print("Note: these skills are not in the dataset and were ignored:", unknown)
    skills = [s for s in skills if s in known_skills]
    if not skills:
        print("No recognised skills. Try e.g. 'python, sql, excel'.")
        return pd.DataFrame()

    user_vec = vectorizer.transform([", ".join(skills)])
    similarity = cosine_similarity(user_vec, role_matrix)[0]

    rows = []
    for i, row in jobs.iterrows():
        required = skill_tokenizer(row["required_skills"])
        have = [s for s in required if s in skills]
        missing = [s for s in required if s not in skills]
        coverage = len(have) / len(required)

        score = 0.6 * similarity[i] + 0.4 * coverage
        if interest and interest.lower() == row["interest_area"].lower():
            score += 0.05
        score = min(score, 1.0)

        rows.append({
            "Role": row["role"],
            "Interest Area": row["interest_area"],
            "Match %": round(score * 100, 1),
            "Skills You Have": ", ".join(have) if have else "-",
            "Skills To Learn Next": ", ".join(missing) if missing else "None - you are ready!",
        })

    result = pd.DataFrame(rows).sort_values("Match %", ascending=False)
    result = result[result["Match %"] > 0].head(top_n)     # hide roles with no overlap at all
    return result.reset_index(drop=True)

## 5. Try it: sample users

In [ ]:
pd.set_option("display.max_colwidth", None)

samples = [
    ("Student A (backend skills)", "python, sql, fastapi, git", None),
    ("Student B (Excel + SQL)", "excel, sql, power bi", None),
    ("Student C (web basics)", "html, css, javascript", None),
    ("Student D (AI interest)", "python, ml, opencv", "Data Science & AI"),
]

for name, skills, interest in samples:
    print("=" * 70)
    print(name, "| skills:", skills, "| interest:", interest)
    display(recommend(skills, interest))

## 6. Your turn
Edit the two lines below and run the cell. Leave `my_interest` as `None` if you have no preference.
Available interest areas are listed in section 2.

In [ ]:
my_skills = "python, sql, excel, git"
my_interest = None          # e.g. "Data Analytics"

result = recommend(my_skills, my_interest)
display(result)

## 7. Visualise the top matches

In [ ]:
if not result.empty:
    plot_df = result.iloc[::-1]          # reverse so the best match is on top
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.barh(plot_df["Role"], plot_df["Match %"], color="#2a6fdb")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Match %")
    ax.set_title("Top career matches")
    for bar, val in zip(bars, plot_df["Match %"]):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2, f"{val}%", va="center")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("recommendation_chart.png", dpi=150)
    plt.show()

## 8. Quick sanity check
A simple test that each role is recommended first when a user has *exactly* that role's skills. This confirms the matching logic works.

In [ ]:
correct = 0
for _, row in jobs.iterrows():
    top = recommend(row["required_skills"], top_n=1)
    if top.loc[0, "Role"] == row["role"]:
        correct += 1
    else:
        print("Mismatch for", row["role"], "->", top.loc[0, "Role"])

print(f"Self-match accuracy: {correct}/{len(jobs)} = {correct / len(jobs):.0%}")

## 9. Impact, limitations and future scope

**SDG 8 link:** Job seekers in smaller cities often do not know which roles fit their skills or what to learn next. This tool gives free, instant, skill-based career guidance, which helps improve employability and supports decent work.

**Limitations**
- The dataset is small (20 roles) and hand-written, so it is a prototype.
- Skills must be typed in; spelling variants outside the alias list are ignored.
- It does not yet consider location, salary or experience level.

**Future scope**
- Expand the dataset using real job postings.
- Add a Streamlit or mobile front end, and regional-language support.
- Use an LLM (for example IBM watsonx) to write a short personalised career summary and learning roadmap.
- Link each missing skill to free courses (for example IBM SkillsBuild).